In [3]:
import os, glob, json
from pathlib import Path
import numpy as np
import pandas as pd


import torch
from botorch.fit import fit_gpytorch_model
from botorch.optim import optimize_acqf
from botorch.acquisition.preference import AnalyticExpectedUtilityOfBestOption
from botorch.models.pairwise_gp import PairwiseGP, PairwiseLaplaceMarginalLogLikelihood

torch.manual_seed(42)
np.random.seed(42)

In [30]:
def load_pairwise(domain, user_id, session_id):
    files = glob.glob(f'../data/{domain}-pairwise/{user_id}-{session_id}/feedbacks/*.json')
    df = pd.DataFrame(columns=[
        'userID',
        'studyID',
        'sessionID',
        'taskID',
        'feedbackID',
        'iteration',
        'completeIteration',
        'incompleteIteration',
        'rating',
        'selection',
        'skip',
        'stop',
        'timePresent',
        'timeLastRating',
        'timeSubmit',
        'comment'
    ])
    for feedback_file in files:
        # load the feedback
        feedback = json.load(open(feedback_file))

        # load corresponding parameters
        param_folder = feedback_file.replace('feedbacks', 'parameters').removesuffix('.json')
        options = [param for param in glob.glob(f'{param_folder}/*.json')]
        selection = feedback['selection']
        selection_id = feedback['rating']

        # if selection id is either 1 or 2 then this means users indicated
        # their preference; if selection id is 3 then it means selected
        # selected both. Otherwise, it means "I don't know".
        if selection_id < 3:
            if Path(options[selection_id-1]).stem != Path(selection).stem:
                options[1], options[0] = options[0], options[1]
        elif selection_id == 3 or selection_id == 0:
            if Path(options[0]).stem != Path(selection).stem:
                options[1], options[0] = options[0], options[1]
        feedback['options'] = options
        feedback['parameters'] = [json.load(open(param)) for param in options]
        feedback['optionsContent'] = [open(param.replace('parameters', 'options').replace('.json', '.txt')).read() for param in options]

        df = pd.concat([df, pd.DataFrame([feedback])], ignore_index=True)
    df.sort_values(by='timeLastRating', inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df

def load_pairwise_data(domain: str, user_id: str, session_id: str, task_id: str):
    df = load_pairwise('text', 'changkun', 'wefoiwejfoij')
    df = df[df['taskID'] == task_id].reset_index(drop=True)

    X = []
    y = []
    for idx, row in df[['taskID', 'parameters', 'rating', 'optionsContent']].iterrows():
        params = row['parameters']
        option1 = [
            params[0]['reduction_ratio'],
            params[0]['top_p'],
            params[0]['temperature'],
            # params[0]['repetition_penalty'],
            params[0]['length_penalty'],
        ]
        option2 = [
            params[1]['reduction_ratio'],
            params[1]['top_p'],
            params[1]['temperature'],
            # params[1]['repetition_penalty'],
            params[1]['length_penalty'],
        ]
        if row['rating'] == 1 or row['rating'] == 2:
            X.append(option1)
            X.append(option2)
            if row['rating'] == 1:
                y.append([2*idx, 2*idx+1])
            elif row['rating'] == 2:
                y.append([2*idx+1, 2*idx])
    return torch.tensor(X, dtype=torch.float64), torch.tensor(y)

In [32]:
def pairwise_bo_estimation(X, y, dim):
    dim = 5
    model = PairwiseGP(X, y)
    mll = PairwiseLaplaceMarginalLogLikelihood(model.likelihood, model)
    mll = fit_gpytorch_model(mll)
    next_X, acq_val = optimize_acqf(
        acq_function=AnalyticExpectedUtilityOfBestOption(pref_model=model),
        bounds=torch.stack([torch.zeros(dim), torch.ones(dim)]),
        q=2,
        num_restarts=3,
        raw_samples=256,
    )
    return next_X, acq_val

X, y = load_pairwise_data('text', 'changkun', 'wefoiwejfoij', '2')
next_X, acq_val = pairwise_bo_estimation(X, y, 5)
next_X

/Users/changkun/miniconda3/envs/bo/lib/python3.9/site-packages/gpytorch/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/changkun/miniconda3/envs/bo/lib/python3.9/site-packages/botorch/fit.py:148: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(w.message, w.category)


tensor([[0.5657, 0.4585, 0.7661, 0.8900, 0.0000],
        [0.2030, 0.6184, 0.0000, 0.1190, 0.6528]])